# Artificial Neural Networks: The Perceptron

**Author:** Waseem Shareef Jamro  
**Date:** 24 November 2026

This notebook studies the single-layer perceptron, implements its learning rule from scratch, visualizes linear decision boundaries, demonstrates the XOR limitation, and tests Scikit-learn's Perceptron on the Iris dataset.

Run all cells from top to bottom.

## 1. Core idea

A perceptron computes a weighted sum and applies a threshold:

\[
z = \mathbf{w}^T\mathbf{x}+b, \qquad \hat y = \begin{cases}1 & z\geq 0\\0 & z<0\end{cases}
\]

For a misclassified training example, the update rule is:

\[
\mathbf{w} \leftarrow \mathbf{w}+\eta(y-\hat y)\mathbf{x},\qquad
b \leftarrow b+\eta(y-\hat y)
\]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)

def step(z):
    return (np.asarray(z) >= 0).astype(int)

def predict_one(w, x, b):
    return int(step(np.dot(w, x) + b))

def train_perceptron(X, y, learning_rate=0.1, max_epochs=100):
    X, y = np.asarray(X, float), np.asarray(y, int)
    w, b, history = np.zeros(X.shape[1]), 0.0, []
    for _ in range(max_epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            error = yi - predict_one(w, xi, b)
            if error:
                w += learning_rate * error * xi
                b += learning_rate * error
                mistakes += 1
        history.append(mistakes)
        if mistakes == 0:
            break
    return w, b, history

print('Setup complete. NumPy:', np.__version__)

## 2. Neuron from scratch and logical gates

The AND and OR truth tables are linearly separable, so a single-layer perceptron can learn them.

In [ ]:
X_logic = np.array([[0,0], [0,1], [1,0], [1,1]])
gates = {'AND': np.array([0,0,0,1]), 'OR': np.array([0,1,1,1])}

for name, y in gates.items():
    w, b, history = train_perceptron(X_logic, y)
    predictions = np.array([predict_one(w, x, b) for x in X_logic])
    print(f'{name}: weights={w}, bias={b:.2f}, predictions={predictions.tolist()}, epochs={len(history)}')
    plt.plot(range(1, len(history)+1), history, marker='o', label=name)

plt.xlabel('Epoch'); plt.ylabel('Misclassifications')
plt.title('Perceptron learning on logical gates')
plt.legend(); plt.grid(True); plt.show()

## 3. Decision boundary

For two input features, the decision boundary is the straight line \(w_1x_1+w_2x_2+b=0\). The perceptron classifies points on the positive side as class 1.

In [ ]:
w, b, _ = train_perceptron(X_logic, gates['OR'])
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
zz = w[0]*xx + w[1]*yy + b
plt.contourf(xx, yy, zz >= 0, alpha=0.25, levels=[-0.5, 0.5, 1.5])
for label in [0, 1]:
    pts = X_logic[gates['OR'] == label]
    plt.scatter(pts[:,0], pts[:,1], label=f'class {label}', s=90)
if abs(w[1]) > 1e-10:
    xline = np.linspace(-0.3, 1.3, 100)
    plt.plot(xline, -(w[0]*xline+b)/w[1], 'k--', label='decision boundary')
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('OR gate decision boundary')
plt.legend(); plt.grid(True); plt.show()

## 4. Why a single perceptron cannot learn XOR

XOR labels alternate across the four corners. No single straight line can separate the two classes. A network with a hidden layer can express XOR as:

\[XOR(x_1,x_2)=AND(OR(x_1,x_2), NAND(x_1,x_2))\]

In [ ]:
y_xor = np.array([0,1,1,0])
def tlu(w, b, x):
    return predict_one(np.asarray(w), np.asarray(x), b)

def xor_network(x):
    hidden_or = tlu([1,1], -0.5, x)
    hidden_nand = tlu([-1,-1], 1.5, x)
    return tlu([1,1], -1.5, [hidden_or, hidden_nand])

xor_results = [xor_network(x) for x in X_logic]
display(pd.DataFrame({'x1': X_logic[:,0], 'x2': X_logic[:,1], 'XOR target': y_xor, 'network output': xor_results}))
print('All XOR outputs correct:', xor_results == y_xor.tolist())

## 5. Iris classification with Scikit-learn

The built-in Iris dataset is used for a binary Setosa-vs-rest experiment and a three-class experiment. The data is split into training and test sets with stratification.

In [ ]:
iris = load_iris()
X = iris.data[:, 2:4]  # petal length and petal width
tasks = {'Setosa vs rest': (iris.target == 0).astype(int), 'Three classes': iris.target}
rows = []
for task, y in tasks.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
    model = Perceptron(random_state=SEED, max_iter=1000)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'Task': task, 'Accuracy': accuracy_score(y_test, pred), 'Macro F1': f1_score(y_test, pred, average='macro')})
    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(f'Confusion matrix — {task}'); plt.show()
display(pd.DataFrame(rows).round(3))

## 6. Synthetic data: perceptron vs. logistic regression

The synthetic dataset can contain overlapping classes. In that case, a perceptron may not reach zero training mistakes; Logistic Regression is included as a reference.

In [ ]:
X_syn, y_syn = make_classification(n_samples=200, n_features=2, n_redundant=0, n_clusters_per_class=1, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X_syn, y_syn, test_size=0.30, random_state=SEED, stratify=y_syn)
w, b, history = train_perceptron(X_train, y_train, learning_rate=1.0, max_epochs=1000)
pred_custom = np.array([predict_one(w, x, b) for x in X_test])
sk = Perceptron(random_state=SEED).fit(X_train, y_train)
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
comparison = pd.DataFrame({
    'Model': ['Hand-built Perceptron', 'Scikit-learn Perceptron', 'Logistic Regression'],
    'Train accuracy': [np.mean([predict_one(w, x, b) for x in X_train] == y_train), sk.score(X_train, y_train), lr.score(X_train, y_train)],
    'Test accuracy': [accuracy_score(y_test, pred_custom), sk.score(X_test, y_test), lr.score(X_test, y_test)]
})
display(comparison.round(3))
plt.plot(range(1, len(history)+1), history)
plt.xlabel('Epoch'); plt.ylabel('Training mistakes'); plt.title('Hand-built perceptron training history')
plt.grid(True); plt.show()

## 7. Summary

- A perceptron combines a weighted sum, bias, and step activation.
- Its decision boundary is linear.
- The perceptron learning rule can converge on linearly separable tasks such as AND and OR.
- A single-layer perceptron cannot represent XOR; a hidden layer solves this limitation.
- Performance on overlapping datasets depends on the data and training procedure; compare models using held-out data.

**Key takeaway:** the perceptron is a useful starting point for understanding neural networks, but more complex patterns require richer models.